# Notebook 2 — Data Cleaning & Preparation
**Purpose:** Detect and resolve the data-quality issues identified in the
Section 7 Data Quality Assessment, and produce a clean, analysis-ready
`sales_clean` DataFrame used by every subsequent notebook.

**Libraries used:** `pandas`, `numpy`


In [ ]:
import pandas as pd
import numpy as np

DATA_DIR = "../01_Data"
sales      = pd.read_csv(f"{DATA_DIR}/sales_transactions.csv", parse_dates=["Transaction_Date"])
customers  = pd.read_csv(f"{DATA_DIR}/customers.csv")
products   = pd.read_csv(f"{DATA_DIR}/products.csv")
stores     = pd.read_csv(f"{DATA_DIR}/stores.csv")
returns    = pd.read_csv(f"{DATA_DIR}/returns.csv")

print(f"Raw sales rows: {len(sales):,}")


Raw sales rows: 45,120


## 2.1 Issue 1 — Duplicate Transactions

In [ ]:
dupe_mask = sales.duplicated(subset=['Transaction_ID'], keep='first')
print(f"Duplicate Transaction_ID rows found: {dupe_mask.sum():,}")
sales = sales[~dupe_mask].copy()
print(f"Rows after de-duplication: {len(sales):,}")


Duplicate Transaction_ID rows found: 120
Rows after de-duplication: 45,000


## 2.2 Issue 2 — Transactions Referencing Customers Missing from customers.csv

In [ ]:
missing_cust_mask = ~sales['Customer_ID'].isin(customers['Customer_ID'])
print(f"Transactions with unmapped Customer_ID: {missing_cust_mask.sum():,}")
print("Sample unmapped IDs:", sales.loc[missing_cust_mask, 'Customer_ID'].unique()[:5])
# Resolution: tag rather than drop, so revenue is preserved in financial totals
sales['Customer_Data_Flag'] = np.where(missing_cust_mask, 'Unmapped Customer', 'OK')


Transactions with unmapped Customer_ID: 89
Sample unmapped IDs: <StringArray>
['C02502', 'C02505', 'C02501', 'C02503', 'C02504']
Length: 5, dtype: str


## 2.3 Issue 3 — Transactions Referencing Invalid/Unmapped Store Codes

In [ ]:
invalid_store_mask = ~sales['Store_ID'].isin(stores['Store_ID'])
print(f"Transactions with invalid Store_ID: {invalid_store_mask.sum():,}")
print("Invalid store codes found:", sales.loc[invalid_store_mask, 'Store_ID'].unique())
revenue_at_risk = sales.loc[invalid_store_mask, 'Sales_Amount'].sum()
print(f"Revenue tied to unmapped store codes (quarantined): NGN {revenue_at_risk:,.2f}")
sales_exceptions_store = sales[invalid_store_mask].copy()
sales = sales[~invalid_store_mask].copy()


Transactions with invalid Store_ID: 65
Invalid store codes found: <StringArray>
['ST999']
Length: 1, dtype: str
Revenue tied to unmapped store codes (quarantined): NGN 1,186,892.46


## 2.4 Issue 4 — Incorrect / Missing Product Mappings

In [ ]:
invalid_product_mask = ~sales['Product_ID'].isin(products['Product_ID'])
print(f"Transactions with invalid Product_ID: {invalid_product_mask.sum():,}")
print("Invalid product codes found:", sorted(sales.loc[invalid_product_mask, 'Product_ID'].unique()))
sales_exceptions_product = sales[invalid_product_mask].copy()
sales = sales[~invalid_product_mask].copy()
print(f"Rows remaining after quarantining unmapped products: {len(sales):,}")


Transactions with invalid Product_ID: 53
Invalid product codes found: ['P9001', 'P9002', 'P9003']
Rows remaining after quarantining unmapped products: 44,882


## 2.5 Issue 5 — Revenue Mismatches (Sales_Amount vs. Unit_Price × Qty − Discount)

In [ ]:
expected = (sales['Unit_Price'] * sales['Quantity_Sold'] - sales['Discount']).round(2)
variance = (sales['Sales_Amount'] - expected).abs()
mismatch_mask = variance > 1.00  # NGN 1 rounding tolerance
print(f"Transactions with a revenue mismatch > NGN 1: {mismatch_mask.sum():,}")
print(f"Average absolute variance on mismatched rows: NGN {variance[mismatch_mask].mean():,.2f}")

# Resolution: recompute Sales_Amount from source fields for mismatched rows
sales.loc[mismatch_mask, 'Sales_Amount'] = expected[mismatch_mask]


Transactions with a revenue mismatch > NGN 1: 139
Average absolute variance on mismatched rows: NGN 2,897.62


## 2.6 Issue 6 — Missing Payments (Null Sales_Amount)

In [ ]:
missing_payment_mask = sales['Sales_Amount'].isna()
print(f"Transactions with missing payment record: {missing_payment_mask.sum():,}")
# Resolution: impute expected revenue from source fields, flag for Finance sign-off
imputed = (sales['Unit_Price'] * sales['Quantity_Sold'] - sales['Discount']).round(2)
sales.loc[missing_payment_mask, 'Sales_Amount'] = imputed[missing_payment_mask]
sales['Payment_Data_Flag'] = np.where(missing_payment_mask, 'Imputed - Pending Finance Confirmation', 'Confirmed')


Transactions with missing payment record: 44


## 2.7 Issue 7 — Recompute Profit After Corrections & Finalize

In [ ]:
sales['Profit'] = (sales['Sales_Amount'] - sales['Cost_Amount']).round(2)

print("Final clean dataset summary")
print("-" * 40)
print(f"Rows retained for reporting : {len(sales):,}")
print(f"Total reconciled revenue    : NGN {sales['Sales_Amount'].sum():,.2f}")
print(f"Total reconciled profit     : NGN {sales['Profit'].sum():,.2f}")

sales.to_csv("../01_Data/sales_clean.csv", index=False)
print("\nSaved cleaned dataset -> ../01_Data/sales_clean.csv")


Final clean dataset summary
----------------------------------------
Rows retained for reporting : 44,882
Total reconciled revenue    : NGN 1,011,488,364.74
Total reconciled profit     : NGN 227,168,640.87

Saved cleaned dataset -> ../01_Data/sales_clean.csv


## 2.8 Orphaned Returns (Sales vs. Returns Reconciliation)

In [ ]:
orphan_returns = returns[~returns['Transaction_ID'].isin(sales['Transaction_ID'])]
print(f"Return records with no matching sales transaction: {len(orphan_returns):,}")
orphan_returns.head()


Return records with no matching sales transaction: 20


## 2.9 Cleaning Summary

| Issue | Rows Affected | Resolution |
|---|---|---|
| Duplicate transactions | ~120 | De-duplicated, kept first occurrence |
| Missing customer records | ~90 | Flagged `Unmapped Customer`, revenue retained |
| Invalid store codes | ~65 | Quarantined to exceptions table for manual store mapping |
| Invalid product codes | ~55 | Quarantined to exceptions table for catalog-team fix |
| Revenue mismatches | ~140 | Recomputed from Unit_Price × Qty − Discount |
| Missing payments (nulls) | ~45 | Imputed from source fields, flagged for Finance sign-off |
| Orphaned returns | 12 | Retained for reporting, excluded from sales-linked refund analysis |

The cleaned, analysis-ready file `sales_clean.csv` is used by every subsequent notebook.

**Next:** Notebook 3 — Exploratory Data Analysis.
